# Two atoms through one cavity — v2

Adds the three things your supervisor asked for:

1. **von Neumann entropy** as the entanglement measure (replacing concurrence)
2. **Truth table** — what gate, if any, does this protocol implement on the two atoms?
3. **Detuning** — what happens when the atoms are detuned from the cavity (Δ > 0)

Protocol: atom 1 does a *half* transit (leaves half its excitation in the cavity),
atom 2 arrives a delay `dt` later and does a *full* transit (picks the photon up).
Result for the `|eg⟩` input is the Bell state `(|eg⟩+|ge⟩)/√2`.

Basis ordering throughout: **subsystem 1 = cavity, 2 = atom1, 3 = atom2**.

In [ ]:
using QuantumOptics
using LinearAlgebra
using Plots
gr()

## Parameters and operators

Note `Δ` is now a *function argument*, not a constant, so we can sweep it.

In [ ]:
const n_max = 2
const g0    = 1.0
const w     = 1.0
const κ     = 0.05
const γ     = 0.02

const v_half = 4 / sqrt(pi) * g0 * w   # atom1: half-transit
const v_full = 2 / sqrt(pi) * g0 * w   # atom2: full-transit

b_cav = FockBasis(n_max)
b_at  = SpinBasis(1//2)

Ic, Ia = one(b_cav), one(b_at)
a   = destroy(b_cav) ⊗ Ia ⊗ Ia
s1m = Ic ⊗ sigmam(b_at) ⊗ Ia
s2m = Ic ⊗ Ia ⊗ sigmam(b_at)
sz1 = Ic ⊗ sigmaz(b_at) ⊗ Ia
sz2 = Ic ⊗ Ia ⊗ sigmaz(b_at)

num  = dagger(a) * a
exc1 = (sz1 + one(sz1)) / 2
exc2 = (sz2 + one(sz2)) / 2

Hc1 = dagger(a) * s1m + a * dagger(s1m)
Hc2 = dagger(a) * s2m + a * dagger(s2m)

J = AbstractOperator[]
κ > 0 && push!(J, sqrt(κ) * a)
γ > 0 && push!(J, sqrt(γ) * s1m)
γ > 0 && push!(J, sqrt(γ) * s2m)
Jdag = dagger.(J)

gpulse(t, t0, v) = g0 * exp(-((v * (t - t0)) / w)^2)

println("setup ok")

## Generalised run

Now takes the detuning `Δ` and the two atoms' initial states as arguments, so the
same function serves the dynamics, the truth table, and the detuning sweep.

In [ ]:
function run_pair(dt; Δ = 0.0, at1 = spinup(b_at), at2 = spindown(b_at), nsteps = 800)
    t01  = 4 * (w / v_half)
    t02  = t01 + dt
    tmax = t02 + 4 * (w / v_full)
    T = range(0, tmax, length = nsteps)

    ψ0 = fockstate(b_cav, 0) ⊗ at1 ⊗ at2
    H0 = Δ * (exc1 + exc2)

    f(t, ρ) = (H0 + gpulse(t, t01, v_half)*Hc1 + gpulse(t, t02, v_full)*Hc2, J, Jdag)
    tout, ρt = timeevolution.master_dynamic(T, ψ0, f)
    return tout, ρt, t01, t02
end

println("run_pair ready")

## 1. von Neumann entropy

`S(ρ) = −Tr(ρ log ρ)`. Computed manually from the eigenvalues (avoids any
API surprises), reported in **bits** so a maximally entangled qubit gives S = 1.

Two quantities worth tracking, and they mean different things:

- `S_atom1` — trace out cavity **and** atom 2, leaving atom 1 alone. For a *pure*
  global state this is the entanglement of atom 1 with everything else.
- `S_atoms` — trace out only the cavity, leaving the two-atom state. This measures
  how *mixed* the atom pair is: nonzero means they are still entangled with the
  cavity, or that loss has degraded them.

**Important caveat to state out loud:** von Neumann entropy of a reduced state is a
clean entanglement measure only when the *global* state is pure. With κ, γ > 0 the
global state is mixed, so `S_atom1` mixes genuine entanglement with classical
mixedness from loss. To read it as pure entanglement, set `κ = γ = 0`.

In [ ]:
function vn_entropy(ρ; bits = true)
    λ = real.(eigvals(Hermitian(Matrix(ρ.data))))
    λ = λ[λ .> 1e-12]
    S = -sum(λ .* log.(λ))
    return bits ? S / log(2) : S
end

S_atom1(ρ) = vn_entropy(ptrace(ρ, [1, 3]))   # keep atom1 only
S_atoms(ρ) = vn_entropy(ptrace(ρ, 1))        # keep both atoms

println("entropy helpers ready")

### Dynamics with entropy (replaces the concurrence plot)

In [ ]:
dt_demo = 3.0
tout, ρt, t01, t02 = run_pair(dt_demo)

Pe1 = real.(expect(exc1, ρt))
Pe2 = real.(expect(exc2, ρt))
nph = real.(expect(num,  ρt))
S1  = [S_atom1(ρ) for ρ in ρt]
S12 = [S_atoms(ρ) for ρ in ρt]
g1v = [gpulse(t, t01, v_half) for t in tout]
g2v = [gpulse(t, t02, v_full) for t in tout]

ptop = plot(tout, g1v, ls=:dash, lc=:steelblue, label="g1(t) atom1", ylabel="coupling",
            title="Two atoms, one cavity — von Neumann entropy")
plot!(ptop, tout, g2v, ls=:dash, lc=:orange, label="g2(t) atom2")

pbot = plot(tout, Pe1, label="atom1 excited", xlabel="time (1/g0)", ylabel="population / entropy (bits)")
plot!(pbot, tout, Pe2, label="atom2 excited")
plot!(pbot, tout, nph, label="cavity <n>")
plot!(pbot, tout, S1,  lc=:black, lw=2, label="S(atom1)  [bits]")
plot!(pbot, tout, S12, lc=:purple, lw=2, ls=:dot, label="S(atom pair)  [bits]")
plot(ptop, pbot, layout=(2,1), size=(860,720))

### Delay sweep with entropy

Same physics as the concurrence sweep, now read through entropy. Run it twice —
once with loss on (as set above) and once with `κ = γ = 0` — to separate genuine
entanglement from loss-induced mixedness.

In [ ]:
delays = range(0.0, 12.0, length = 40)
S1_final, S12_final, Pe2_final = Float64[], Float64[], Float64[]
for dt in delays
    _, ρ, _, _ = run_pair(dt)
    push!(S1_final,  S_atom1(ρ[end]))
    push!(S12_final, S_atoms(ρ[end]))
    push!(Pe2_final, real(expect(exc2, ρ[end])))
end

plot(delays, S1_final, m=:circle, lw=2, label="S(atom1)  [bits]",
     xlabel="inter-atom delay dt (1/g0)", ylabel="value",
     title="Entropy vs delay — the cavity's memory window")
plot!(delays, S12_final, m=:diamond, lw=2, label="S(atom pair)  [bits]")
plot!(delays, Pe2_final, m=:square, label="excitation relayed to atom2")

## 2. Truth table — is this actually a gate?

A two-qubit gate must map the atomic subspace back onto itself for **every** input:
whatever the cavity does mid-protocol, it has to end in vacuum, or the atoms have
leaked information into it and the operation is not unitary on the atoms.

So we feed in all four computational-basis inputs `|gg⟩, |ge⟩, |eg⟩, |ee⟩` and record
(a) the final atomic populations, and (b) how many photons are left in the cavity.

**Read the leftover-photon column first** — it is the pass/fail test.

In [ ]:
ket_at(c) = c == 'e' ? spinup(b_at) : spindown(b_at)
at2ket(s) = ket_at(s[1]) ⊗ ket_at(s[2])
pop_at(ρat, s) = real(dagger(at2ket(s)) * (ρat * at2ket(s)))

labels = ["gg", "ge", "eg", "ee"]

function truth_table(; Δ = 0.0, dt = 3.0)
    println("input |  gg     ge     eg     ee   | photons left | in-subspace")
    println("------+------------------------------+--------------+------------")
    for s in labels
        _, ρt, _, _ = run_pair(dt; Δ = Δ, at1 = ket_at(s[1]), at2 = ket_at(s[2]))
        ρf  = ρt[end]
        ρat = ptrace(ρf, 1)
        pops = [pop_at(ρat, o) for o in labels]
        nleft = real(expect(num, ρf))
        println("  ", s, "  | ", join([string(round(p, digits=3), "  ") for p in pops]),
                " |    ", round(nleft, digits=3), "     |   ", round(sum(pops), digits=3))
    end
end

truth_table(Δ = 0.0, dt = 3.0)

In [ ]:
Pvac(ρ) = real(ptrace(ρ, [2,3]).data[1,1])   # cavity reduced state, vacuum population

function truth_table2(; Δ = 0.0, dt = 3.0)
    println("input |  gg     ge     eg     ee   | P(cav vacuum) | S(pair) bits")
    println("------+------------------------------+---------------+-------------")
    for s in labels
        _, ρt, _, _ = run_pair(dt; Δ = Δ, at1 = ket_at(s[1]), at2 = ket_at(s[2]))
        ρf   = ρt[end] 
        ρat  = ptrace(ρf, 1)
        pops = [pop_at(ρat, o) for o in labels]
        println("  ", s, "  | ", join([string(round(p, digits=3), "  ") for p in pops]),
                " |     ", round(Pvac(ρf), digits=3),
                "     |    ", round(S_atoms(ρf), digits=3))
    end
end

truth_table2()

### How to read that table

- **`|gg⟩`** — nothing to exchange, stays `|gg⟩`, cavity empty. Fine.
- **`|eg⟩`** — this is our Bell case: population splits ~50/50 between `eg` and `ge`,
  cavity empty. (Populations alone cannot distinguish the entangled superposition
  from a classical mixture — that is what the entropy is for.)
- **`|ge⟩`** — atom 1 is in the ground state so its half-transit does nothing; then
  atom 2 does a *full* transit and **dumps its photon into the cavity**. Expect
  ≈1 photon left. The state has left the atomic subspace.
- **`|ee⟩`** — two excitations, √2-enhanced Rabi, no clean π condition. Expect
  leftover photons here too.

So the honest conclusion: **this protocol is not a universal two-qubit gate.** It is
an excellent *entangler for one specific input*. The asymmetry is baked in — atom 1
is assigned a half-transit and atom 2 a full transit, so the operation is not
symmetric under swapping which atom carries the excitation.

That is a genuine result worth reporting, and it points straight at the fix below.

## 3. Detuning Δ > 0

Two regimes, and they are qualitatively different.

**(a) Detuning the existing resonant protocol** — the transits no longer meet the
π/2 and π conditions, so the exchange becomes partial and the entanglement degrades.
This is the expected "detuning spoils it" answer.

In [ ]:
deltas = range(0.0, 3.0, length = 25)
S1_vs_Δ, n_vs_Δ = Float64[], Float64[]
for Δ in deltas
    _, ρ, _, _ = run_pair(3.0; Δ = Δ)
    push!(S1_vs_Δ, S_atom1(ρ[end]))
    push!(n_vs_Δ,  real(expect(num, ρ[end])))
end

plot(deltas, S1_vs_Δ, m=:circle, lw=2, label="S(atom1) [bits]",
     xlabel="detuning Δ / g0", ylabel="value",
     title="Resonant protocol under detuning")
plot!(deltas, n_vs_Δ, m=:square, label="photons left in cavity")

**(b) The interesting regime — dispersive, Δ ≫ g0.**

When the atoms are far detuned they can no longer *really* absorb or emit a photon.
But they can still exchange excitation through a **virtual** photon, giving an
effective atom–atom exchange (XY) interaction of strength `J ≈ g0²/Δ`, with the
cavity staying essentially empty the whole time.

That matters because it is the standard route to a genuine cavity-mediated two-qubit
gate: the interaction is excitation-conserving, it closes on the atomic subspace, and
because the photon is virtual it is far less sensitive to cavity loss κ.

Two changes are needed versus the sequential protocol:
- the atoms must be in the cavity **at the same time** (same pulse centre), and
- the transit must be much **slower**, since J is smaller by a factor g0/Δ. The
  velocity below is set from the full-exchange condition; you may need to tune it.

In [ ]:
function run_dispersive(Δ; v = nothing, at1 = spinup(b_at), at2 = spindown(b_at), nsteps = 1500)
    v === nothing && (v = g0^2 * w * sqrt(2/pi) / Δ)     # full-exchange estimate
    t0   = 4 * (w / v)
    tmax = 8 * (w / v)
    T = range(0, tmax, length = nsteps)

    ψ0 = fockstate(b_cav, 0) ⊗ at1 ⊗ at2
    H0 = Δ * (exc1 + exc2)
    f(t, ρ) = (H0 + gpulse(t, t0, v) * (Hc1 + Hc2), J, Jdag)
    tout, ρt = timeevolution.master_dynamic(T, ψ0, f)
    return tout, ρt, v
end

Δd = 8.0
tout_d, ρd, vd = run_dispersive(Δd)
println("dispersive run: Δ = ", Δd, ", transit velocity = ", round(vd, digits=4))

Pe1d = real.(expect(exc1, ρd))
Pe2d = real.(expect(exc2, ρd))
nd   = real.(expect(num,  ρd))
S1d  = [S_atom1(ρ) for ρ in ρd]

plot(tout_d, Pe1d, lw=2, label="atom1 excited", xlabel="time (1/g0)",
     ylabel="population / entropy (bits)",
     title="Dispersive regime: virtual-photon exchange (Δ = "*string(Δd)*")")
plot!(tout_d, Pe2d, lw=2, label="atom2 excited")
plot!(tout_d, nd,   lw=2, label="cavity <n>  (should stay small)")
plot!(tout_d, S1d,  lc=:black, lw=2, label="S(atom1) [bits]")

**What to look for:** excitation should slosh from atom 1 to atom 2 while
`⟨n⟩` stays near zero — the photon is never really there. Stop the interaction at a
*half* exchange and you get the maximally entangled state (S(atom1) → 1 bit) with the
cavity empty; a *full* exchange gives an iSWAP-like operation.

If `⟨n⟩` is not small, Δ is not large enough relative to g0. If nothing moves at all,
the transit is too fast — lower `v` (pass it explicitly, e.g. `run_dispersive(8.0, v=0.02)`).

### Truth table in the dispersive regime

Worth running for comparison — because this interaction is symmetric between the two
atoms and conserves excitation number, the leftover-photon column should be much
closer to zero across *all four* inputs than it was for the sequential protocol.

In [ ]:
function truth_table_dispersive(Δ)
    println("input |  gg     ge     eg     ee   | photons left | in-subspace")
    println("------+------------------------------+--------------+------------")
    for s in labels
        _, ρt, _ = run_dispersive(Δ; at1 = ket_at(s[1]), at2 = ket_at(s[2]))
        ρf  = ρt[end]
        ρat = ptrace(ρf, 1)
        pops = [pop_at(ρat, o) for o in labels]
        nleft = real(expect(num, ρf))
        println("  ", s, "  | ", join([string(round(p, digits=3), "  ") for p in pops]),
                " |    ", round(nleft, digits=3), "     |   ", round(sum(pops), digits=3))
    end
end

truth_table_dispersive(8.0)